# 01 - Data processing

Load CRITEO-UPLIFTv2.1, check shape, define the causal contract
(`X = f0..f11`, `T = treatment`, `Y = conversion`), and build a train/
validation split.

**On Kaggle:** attach a dataset containing `criteo-uplift-v2.1.csv` before
running this notebook.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    CATEGORICAL_FEATURES,
    CONTINUOUS_FEATURES,
    FEATURE_COLUMNS,
    PRIMARY_OUTCOME,
    SECONDARY_OUTCOME,
    TREATMENT_COLUMN,
    basic_summary,
    load_csv,
    save_parquet,
)
from src.preprocessing import train_validation_split

# Kaggle input path if attached as a dataset; otherwise a local data/raw/ copy.
CSV_CANDIDATES = [
    Path("/kaggle/input/criteo-uplift-v2-1/criteo-uplift-v2.1.csv"),
    REPO_ROOT / "data" / "raw" / "criteo-uplift-v2.1.csv",
]
CSV_PATH = next((p for p in CSV_CANDIDATES if p.exists()), CSV_CANDIDATES[-1])
CSV_PATH

## Load and validate shape

In [ ]:
frame = load_csv(CSV_PATH)
frame.shape

In [ ]:
summary = basic_summary(frame)
summary

## Feature contract

`X` is exactly the ordered feature set `f0`-`f11`. All twelve are stored as
`float64`, but that's a storage detail, not their semantic type: per the
CRITEO-UPLIFTv2.1 publisher documentation, `f0`, `f2`, `f7`, `f10` are
genuinely **continuous**, and `f1`, `f3`, `f4`, `f5`, `f6`, `f8`, `f9`, `f11`
are **categorical numeric tokens** with no ordinal meaning -- there is no
sense in which category `"14.0"` is "between" `"3.0"` and `"27.0"`.

`T = treatment` is the (randomized) assignment indicator. `Y = conversion` is
the primary outcome; `visit` is an optional secondary outcome. `exposure` is
post-assignment information and never enters `X`.

In [ ]:
print("continuous:", CONTINUOUS_FEATURES)
print("categorical:", CATEGORICAL_FEATURES)

X = frame[list(FEATURE_COLUMNS)]
T = frame[TREATMENT_COLUMN]
Y = frame[PRIMARY_OUTCOME]
visit = frame[SECONDARY_OUTCOME]

print("treatment rate:", T.mean().round(4))
print("conversion rate:", Y.mean().round(6))
print("visit rate:", visit.mean().round(6))

## Basic distribution check

A quick sanity look at the continuous features and the categorical
cardinalities -- not a full audit, just enough to catch an obviously broken
load.

In [ ]:
frame[list(CONTINUOUS_FEATURES)].describe()

In [ ]:
frame[list(CATEGORICAL_FEATURES)].nunique().sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(CONTINUOUS_FEATURES), figsize=(16, 3))
for ax, feature in zip(axes, CONTINUOUS_FEATURES):
    ax.hist(frame[feature], bins=50)
    ax.set_title(feature)
fig.tight_layout()

## Train / validation split

One seeded, joint-`(treatment, conversion)`-stratified split, so both arms
and both outcome classes are represented in both halves. This is a plain
train/validation split for model development -- there's no separate held-out
test partition or freeze gate in this project's simplified design.

In [ ]:
train_frame, val_frame = train_validation_split(frame, validation_fraction=0.15, seed=42)
print("train:", train_frame.shape, "validation:", val_frame.shape)
print("train treatment rate:", train_frame[TREATMENT_COLUMN].mean().round(4))
print("validation treatment rate:", val_frame[TREATMENT_COLUMN].mean().round(4))

## Save processed Parquet (optional)

Useful so notebooks 02-04 don't need to reload/re-split the raw CSV.

In [ ]:
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
save_parquet(train_frame, PROCESSED_DIR / "train.parquet")
save_parquet(val_frame, PROCESSED_DIR / "validation.parquet")
print("saved to", PROCESSED_DIR)

## Next

Continue with `02_baseline_models.ipynb` (Response LightGBM), then
`03_uplift_models.ipynb` (T-Learner, X-Learner) and `04_causal_forest.ipynb`.